Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\colab\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [5]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [6]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [7]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [8]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [9]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [10]:
datosNormalizados.shape

(43800, 6)

In [11]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [12]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [13]:
futuros = 24
pasados  = 12

In [14]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [15]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 6)
Dimensiones de Y: (43765, 1)


In [16]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [17]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (43765, 72)


Se dividen nuevamente los conjuntos de datos

In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 72)
Las dimensiones de testX son:  (8797, 72)
Las dimensiones de valX son:  (4333, 72)


In [19]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [20]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [21]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [22]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [23]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

958/958 - 25s - 26ms/step - ia: 0.2798 - loss: 3.7096 - mae: 1.2781 - rmse: 1.7388 - smape: 1.4483 - val_ia: 0.2520 - val_loss: 0.8778 - val_mae: 0.7035 - val_rmse: 0.7990 - val_smape: 1.4732

Epoch 2/128                                           

958/958 - 8s - 8ms/step - ia: 0.2924 - loss: 1.2283 - mae: 0.8283 - rmse: 1.0863 - smape: 1.4558 - val_ia: 0.2471 - val_loss: 0.8564 - val_mae: 0.6960 - val_rmse: 0.7814 - val_smape: 1.5313

Epoch 3/128                                           

958/958 - 7s - 8ms/step - ia: 0.2832 - loss: 1.0564 - mae: 0.7700 - rmse: 1.0076 - smape: 1.4796 - val_ia: 0.2466 - val_loss: 0.8457 - val_mae: 0.6860 - val_rmse: 0.7699 - val_smape: 1.5145

Epoch 4/128                                           

958/958 - 8s - 8ms/step - ia: 0.2814 - loss: 0.9885 - mae: 0.7439 - rmse: 0.9753 - smape: 1.4904 - val_ia: 0.2481 - val_loss: 0.8200 - val_mae: 0.6796 - val_rmse: 0.7642 - val_smape: 1.4673

Epoch 5/12

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                         

120/120 - 15s - 127ms/step - ia: 0.3598 - loss: 0.8335 - mae: 0.6736 - rmse: 0.9098 - smape: 1.3268 - val_ia: 0.3562 - val_loss: 0.7574 - val_mae: 0.6330 - val_rmse: 0.8023 - val_smape: 1.2286

Epoch 2/16                                                                         

120/120 - 2s - 14ms/step - ia: 0.4181 - loss: 0.7758 - mae: 0.6427 - rmse: 0.8787 - smape: 1.2385 - val_ia: 0.3557 - val_loss: 0.7586 - val_mae: 0.6494 - val_rmse: 0.8125 - val_smape: 1.2876

Epoch 3/16                                                                         

120/120 - 1s - 6ms/step - ia: 0.4345 - loss: 0.7541 - mae: 0.6311 - rmse: 0.8653 - smape: 1.2170 - val_ia: 0.3601 - val_loss: 0.7759 - val_mae: 0.6552 - val_rmse: 0.8229 - val_smape: 1.2848

Epoch 4/16                                                                         

120/120 - 1s - 5ms/step - ia: 0.4473 - loss: 0.7383 - mae: 0.6231 - rmse: 0.8569 - 

...

Epoch 1/16                                                                          

120/120 - 22s - 183ms/step - ia: 0.1888 - loss: 1.0335 - mae: 0.7627 - rmse: 1.0130 - smape: 1.5795 - val_ia: 0.2582 - val_loss: 0.9137 - val_mae: 0.7121 - val_rmse: 0.8711 - val_smape: 1.7054

Epoch 2/16                                                                          

120/120 - 3s - 23ms/step - ia: 0.2857 - loss: 0.9336 - mae: 0.7195 - rmse: 0.9629 - smape: 1.4264 - val_ia: 0.3261 - val_loss: 0.8051 - val_mae: 0.6492 - val_rmse: 0.8179 - val_smape: 1.2577

Epoch 3/16                                                                          

120/120 - 3s - 29ms/step - ia: 0.3613 - loss: 0.8880 - mae: 0.6985 - rmse: 0.9407 - smape: 1.3159 - val_ia: 0.3254 - val_loss: 0.7887 - val_mae: 0.6567 - val_rmse: 0.8176 - val_smape: 1.2850

Epoch 4/16                                                                          

120/120 - 3s - 25ms/step - ia: 0.3739 - loss: 0.8664 - mae: 0.6874 - rmse: 0.9

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                         

240/240 - 14s - 59ms/step - ia: 0.1363 - loss: 35.3191 - mae: 3.9799 - rmse: 5.9065 - smape: 1.6064 - val_ia: 0.1952 - val_loss: 9.0454 - val_mae: 2.5674 - val_rmse: 2.8357 - val_smape: 1.6241

Epoch 2/128                                                                         

240/240 - 4s - 16ms/step - ia: 0.1419 - loss: 31.1603 - mae: 3.7469 - rmse: 5.5425 - smape: 1.6016 - val_ia: 0.2061 - val_loss: 7.5267 - val_mae: 2.3297 - val_rmse: 2.5941 - val_smape: 1.6045

Epoch 3/128                                                                         

240/240 - 3s - 13ms/step - ia: 0.1489 - loss: 27.3961 - mae: 3.5446 - rmse: 5.1947 - smape: 1.5897 - val_ia: 0.2167 - val_loss: 6.3000 - val_mae: 2.1184 - val_rmse: 2.3797 - val_smape: 1.5864

Epoch 4/128                                                                         

240/240 - 3s - 12ms/step - ia: 0.1563 - loss: 24.3831 - mae: 3.3484 - rmse: 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/32                                                                          

3830/3830 - 35s - 9ms/step - ia: 0.3704 - loss: 0.8909 - mae: 0.6963 - rmse: 0.8827 - smape: 1.3258 - val_ia: 0.2053 - val_loss: 0.7552 - val_mae: 0.6416 - val_rmse: 0.6829 - val_smape: 1.2583

Epoch 2/32                                                                          

3830/3830 - 20s - 5ms/step - ia: 0.3831 - loss: 0.8249 - mae: 0.6672 - rmse: 0.8502 - smape: 1.3052 - val_ia: 0.2032 - val_loss: 0.7515 - val_mae: 0.6442 - val_rmse: 0.6834 - val_smape: 1.2926

Epoch 3/32                                                                          

3830/3830 - 22s - 6ms/step - ia: 0.3859 - loss: 0.8161 - mae: 0.6644 - rmse: 0.8486 - smape: 1.2956 - val_ia: 0.2039 - val_loss: 0.7511 - val_mae: 0.6403 - val_rmse: 0.6801 - val_smape: 1.2618

Epoch 4/32                                                                          

3830/3830 - 40s - 10ms/step - ia: 0.3899 - loss: 0.8112 - mae: 0.6622 - rm

In [24]:
print(best)

{'activation': 1, 'batch': 0, 'dropout': 0.1, 'epochs': 2, 'layers': 4.0, 'learning_rate': 0.0007115052124615948, 'units': 0}
